In [5]:
import pandas as pd
import numpy as np
import torch as t
import torch.nn as nn
from torch.optim import Adam as Adam
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
all_training_paths = [
    "data/5/exp05H20140926_10h50.csv",
    "data/5/exp05H20141001_10h05.csv",
    "data/5/exp05H20141003_15h00.csv",
    "data/5/exp05H20141008_15h30.csv",
    "data/5/exp05H20141010_10h38.csv",
    "data/5/exp05H20141015_12h50.csv",
    "data/5/exp05H20141022_11h45.csv",
    "data/5/exp05H20141023_16h20.csv",
    "data/5/exp05H20141029_14h20.csv",
]

In [ ]:
feature_functions= [
    
]

In [3]:
def split_at_nans_vectorized(data: pd.DataFrame) -> list[pd.DataFrame]:
    """Splits data into contiguous chunks, breaking at any NaN values."""
    
    # 1. Create a boolean mask: True if any NaN exists in the row
    is_nan = data.isna().any(axis=1)
    
    # 2. Create unique group IDs. 
    # The cumulative sum increases by 1 every time it hits a NaN row.
    # This automatically assigns the same integer to contiguous blocks of valid data.
    block_ids = is_nan.cumsum()
    
    # 3. Filter out the actual NaN rows from both the data and the IDs
    valid_data = data[~is_nan]
    valid_block_ids = block_ids[~is_nan]
    
    # 4. Group by the IDs and extract the DataFrames as a list
    data_blocks = [group for _, group in valid_data.groupby(valid_block_ids)]
    
    return data_blocks

In [6]:
def create_sequences(data: np.ndarray, window_size = 30 ) -> tuple[np.ndarray, np.ndarray]:

    shape = data.shape

    X = np.zeros((shape[1], shape[0]-window_size, window_size))
    y = np.zeros((shape[1], shape[0]-window_size))

    for c in range(shape[1]):

        for i in range(len(data) - window_size):

            X[c,i] = data[i : i + window_size, c]
            
            y[c,i] = data[i + window_size, c]

    X = X.transpose(1, 2, 0)
    
    y = y.transpose(1, 0)

    return X, y

In [ ]:
def fit_scaler(training_paths: list[str], feature_functions: list[function]) -> StandardScaler:
    scaler = StandardScaler()

    for path in training_paths:
        
        file_dfs = []

        raw_file_pd = pd.read_csv(f'../{path}')

        nan_free_blocks = split_at_nans_vectorized(raw_file_pd)

        for block in nan_free_blocks:
            block_dfs = []

            for function in feature_functions:
                    
                block_dfs.append(function(block))

            block_attribute_df = pd.concat(block_dfs, axis = 1)

            file_dfs.append(block_attribute_df)

    attribute_df = pd.concat(file_dfs, axis = 0)

    scaler = scaler.fit(attribute_df)

    return scaler

In [ ]:
def create_training_sequences(training_paths: pd.DataFrame, feature_functions: list[function], scaler: StandardScaler)-> tuple[np.ndarray, np.ndarray] :
    for path in training_paths:
        
        file_dfs = []

        raw_file_pd = pd.read_csv(f'../{path}')

        nan_free_blocks = split_at_nans_vectorized(raw_file_pd)

        for block in nan_free_blocks:
            block_dfs = []

            for function in feature_functions:
                    
                block_dfs.append(function(block))

            block_attribute_df = pd.concat(block_dfs, axis = 1)

            file_dfs.append(block_attribute_df)

    attribute_df = pd.concat(file_dfs, axis = 0)

    attribute_np = attribute_df.to_numpy()

    attribute_np_scaled = scaler.transform(attribute_np)

    attribute_np_sequences = create_sequences(attribute_np_scaled, window_size = 30)

    return attribute_np_sequences

In [ ]:
def sequences_to_train_val(attribute_np_sequences:tuple[np.ndarray, np.ndarray]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    x, y = attribute_np_sequences

    y = y[:,:2] #Only predicting v and av

    X_train, X_val, y_train, y_val = train_test_split(
        x, y,
        test_size=0.2,
        random_state=42
    )

    return X_train, X_val, y_train, y_val